In [0]:
# Create a text widget named "catalog" with default value "new_catalog"
dbutils.widgets.text("catalog", "new_catalog")

# Retrieve the value entered in the "catalog" widget, strip whitespace, and store in variable Catalog1
Catalog1 = dbutils.widgets.get("catalog").strip()

# Create a text widget named "schema" with default value "default_schema"
dbutils.widgets.text("schema", "default_schema")

# Retrieve the value entered in the "schema" widget, strip whitespace, and store in variable Schema1
Schema1 = dbutils.widgets.get("schema").strip()

In [0]:
import json

# Run the common configuration notebook with a 360-second timeout
# Pass in dynamic parameters for catalog and schema (from widgets)
json_obj = dbutils.notebook.run(
    "/Workspace/Users/viggneshwar@gmail.com/databricks/Logistics/Project/Generic_Function/common_config_nb",
    360,
    {"catalog_new": Catalog1, "schema_new": Schema1}
)

# Parse the JSON string returned by the notebook into a Python dictionary
config_dict = json.loads(json_obj)

# Extract key configuration values from the dictionary
bronze_path = config_dict["bronze_path"]   # Path for bronze layer data
silver_path = config_dict["silver_path"]   # Path for silver layer data
silver_db   = config_dict["silver_db"]     # Database/schema for silver layer tables
gold_path   = config_dict["gold_path"]     # Path for gold layer data
gold_db     = config_dict["gold_db"]       # Database/schema for gold layer tables

In [0]:
%run "/Workspace/Users/viggneshwar@gmail.com/databricks/Logistics/Project/Generic_Function/generic_functions_nb"

In [0]:

%run "/Workspace/Users/viggneshwar@gmail.com/databricks/Logistics/Project/Generic_Function/business_specific_functions"

In [0]:
# Read all records from the silver staff table
staff_temp = spark.sql(f"SELECT * FROM {silver_db}.staff_silver_tbl")

# Write the DataFrame into the gold staff table
# - mode="overwrite" ensures the gold table is refreshed with the latest silver data
# - file_type='table' indicates you’re writing to a managed Delta table
# - table name is dynamically built using the gold database config
write_file(
    mode="overwrite",
    file_type="table",
    table=f"{gold_db}.staff_gold_tbl",
    df=staff_temp
)

In [0]:
print(gold_db)

In [0]:
# Apply gold curation logic on the logistics shipment silver table
# This function typically standardizes, enriches, and selects business-ready fields
logistics_shipment_temp = logistics_shipment_gold_curation(
    f"{silver_db}.logistics_shipment_silver_tbl"
)

# Write the curated DataFrame into the gold layer table
# - mode="overwrite" ensures the gold table is refreshed with the latest curated data
# - file_type='table' indicates you’re writing to a managed Delta table
# - table name is dynamically built using the gold database config
write_file(
    mode="overwrite",
    file_type="table",
    table=f"{gold_db}.logistics_shipment_gold_curated_tbl",
    df=logistics_shipment_temp
)